# Out-of-Domain Agent Trace Evaluation: BFCL & ToolBench
### Comparative Empirical Assessment: `google/gemma-2-2b-it` vs. `orangefabercastell/gemma-2-2b-it-pi-mono-sft`

**Streamlined 4-Stage Architecture:**
1. **Cell 1:** Hardware Diagnostics, CPU/GPU Telemetry & Hugging Face Gated Authentication
2. **Cell 2:** Ingestion & Schema Alignment (100 BFCL + 100 ToolBench = 200 Master Samples)
3. **Cell 3:** Unified Evaluation Engine with Live VRAM Tracking & Guaranteed Teardown
4. **Cell 4:** Comparative Scorecard, Styled Pandas Display & JSON/CSV Artifact Export

In [ ]:
# [Cell 1] Hardware Telemetry, Environment Diagnostics & Gated HF Authentication
import os
import sys
import time
import gc
import json
import re
import urllib.request
import psutil
import pandas as pd
import numpy as np
import torch
from tqdm.auto import tqdm

# 1. Authenticate with Hugging Face (Required for gated google/gemma-2-2b-it)
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
except Exception:
    pass

if not hf_token:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    import huggingface_hub
    huggingface_hub.login(token=hf_token)
    print("Hugging Face authentication successful via token.")
else:
    print("WARNING: HF_TOKEN not detected in Kaggle Secrets or environment.")
    print("If accessing gated models (google/gemma-2-2b-it), add HF_TOKEN to Kaggle Secrets (Add-ons -> Secrets).")

# 2. Hardware Telemetry Helper
def get_system_telemetry():
    cpu_pct = psutil.cpu_percent(interval=None)
    vmem = psutil.virtual_memory()
    ram_used_gb = vmem.used / (1024**3)
    ram_total_gb = vmem.total / (1024**3)
    
    gpus = []
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            prop = torch.cuda.get_device_properties(i)
            alloc_gb = torch.cuda.memory_allocated(i) / (1024**3)
            res_gb = torch.cuda.memory_reserved(i) / (1024**3)
            total_gb = prop.total_memory / (1024**3)
            gpus.append({
                "id": i,
                "name": prop.name,
                "alloc_gb": alloc_gb,
                "res_gb": res_gb,
                "total_gb": total_gb,
                "util_pct": (alloc_gb / total_gb) * 100
            })
    return {
        "cpu_pct": cpu_pct,
        "ram_used_gb": ram_used_gb,
        "ram_total_gb": ram_total_gb,
        "ram_pct": vmem.percent,
        "gpus": gpus
    }

def print_telemetry(step_name: str):
    tel = get_system_telemetry()
    width = 75
    print("\n" + "=" * width)
    print(f" [SYSTEM MONITOR] {step_name.upper()}")
    print("-" * width)
    print(f" CPU: {tel['cpu_pct']:5.1f}% | Host RAM: {tel['ram_used_gb']:.2f}/{tel['ram_total_gb']:.2f} GB ({tel['ram_pct']:.1f}%)")
    if tel['gpus']:
        for g in tel['gpus']:
            print(f" GPU {g['id']} ({g['name']}): Alloc: {g['alloc_gb']:.2f} GB | Cache: {g['res_gb']:.2f} GB | Cap: {g['total_gb']:.2f} GB ({g['util_pct']:.1f}%)")
    else:
        print(" GPU: CUDA not available")
    print("=" * width + "\n")

print_telemetry("Environment Initialized")


In [ ]:
# [Cell 2] Ingestion & Standardization: BFCL (100) + ToolBench (100) -> Master Suite
print_telemetry("Dataset Ingestion Starting")

BFCL_URL = "https://huggingface.co/datasets/gorilla-llm/Berkeley-Function-Calling-Leaderboard/raw/main/BFCL_v3_simple.json"
BFCL_ANS_URL = "https://huggingface.co/datasets/gorilla-llm/Berkeley-Function-Calling-Leaderboard/raw/main/possible_answer/BFCL_v3_simple.json"
TOOLBENCH_PARQUET_URL = "https://huggingface.co/datasets/tuandunghcmut/toolbench-v1/resolve/main/benchmark/g1_instruction-00000-of-00001.parquet"

def fetch_json_lines(url: str, limit: int = 100):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    raw = urllib.request.urlopen(req).read().decode('utf-8')
    lines = [json.loads(line) for line in raw.strip().split('\n') if line.strip()]
    return lines[:limit]

# Evaluation Sample Volume Config:
# SMOKE_TEST = True -> 40 samples total (20 BFCL + 20 ToolBench, ~3 min runtime)
# SMOKE_TEST = False -> 200 samples total (100 BFCL + 100 ToolBench, ~12 min runtime)
SMOKE_TEST = True
SAMPLE_LIMIT = 20 if SMOKE_TEST else 100

print(f"Evaluation Mode: {'SMOKE TEST (40 Samples)' if SMOKE_TEST else 'FULL BENCHMARK (200 Samples)'}")
print(f"Ingesting {SAMPLE_LIMIT} samples from BFCL and {SAMPLE_LIMIT} from ToolBench...")

# 1. Process samples from BFCL
print("Downloading BFCL questions and answers...")
bfcl_questions = fetch_json_lines(BFCL_URL, limit=SAMPLE_LIMIT)
bfcl_answers = {item["id"]: item.get("ground_truth", []) for item in fetch_json_lines(BFCL_ANS_URL, limit=SAMPLE_LIMIT)}

bfcl_samples = []
for item in tqdm(bfcl_questions, desc="Standardizing BFCL", unit="item"):
    q_id = item["id"]
    user_query = item["question"][0][0]["content"]
    tool_spec = item["function"][0]
    expected_tool = tool_spec["name"].strip().lower()
    expected_args = bfcl_answers.get(q_id, [{}])[0].get(expected_tool, {}) if q_id in bfcl_answers else {}
    
    prompt = (
        f"You are an AI agent with access to tools.\n"
        f"Tool definition:\n{json.dumps(tool_spec, indent=2)}\n\n"
        f"User request: {user_query}\n"
        f"Respond with the exact function call required to solve this request."
    )
    bfcl_samples.append({
        "id": q_id,
        "benchmark": "BFCL",
        "prompt": prompt,
        "expected_tool": expected_tool,
        "expected_args": expected_args
    })

# 2. Process samples from ToolBench
print("Downloading samples from ToolBench...")
local_tb_path = "toolbench_sample.parquet"
if not os.path.exists(local_tb_path):
    urllib.request.urlretrieve(TOOLBENCH_PARQUET_URL, local_tb_path)

tb_df = pd.read_parquet(local_tb_path).head(SAMPLE_LIMIT)
toolbench_samples = []
for idx, row in tqdm(tb_df.iterrows(), total=len(tb_df), desc="Standardizing ToolBench", unit="item"):
    query = row.get("query", "")
    tool_name = str(row.get("api_name", row.get("tool_name", "api_call"))).strip().lower()
    prompt = (
        f"You are an autonomous AI agent integrated into a developer runtime.\n"
        f"Target tool: {tool_name}\n"
        f"User request: {query}\n"
        f"Respond with the structured tool invocation to perform this action."
    )
    toolbench_samples.append({
        "id": f"tb_{idx}",
        "benchmark": "ToolBench",
        "prompt": prompt,
        "expected_tool": tool_name,
        "expected_args": {}
    })

MASTER_EVAL_SUITE = bfcl_samples + toolbench_samples
print(f"Master Evaluation Suite ready: {len(MASTER_EVAL_SUITE)} samples total (100 BFCL + 100 ToolBench).")
print_telemetry("Dataset Ingestion Complete")


In [ ]:
# [Cell 3] Unified Evaluation Engine: Base vs. SFT Model Evaluation & Teardown
from transformers import AutoModelForCausalLM, AutoTokenizer

def parse_agent_call(text: str):
    # XML style: <tool_call>...</tool_call>
    xml_m = re.search(r"<tool_call>\s*(.*?)\s*</tool_call>", text, re.DOTALL)
    if xml_m:
        try:
            return json.loads(xml_m.group(1))
        except Exception:
            pass

    # Markdown JSON block: ```json ... ```
    json_m = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if json_m:
        try:
            return json.loads(json_m.group(1))
        except Exception:
            pass

    # Function syntax: tool_name(param=...)
    fn_m = re.search(r'([a-zA-Z0-9_\.]+)\s*\((.*?)\)', text)
    if fn_m:
        return {"name": fn_m.group(1), "arguments": fn_m.group(2)}

    # Direct JSON dictionary with tool/function key
    dict_m = re.search(r'\{[^{}]*"(?:name|function|tool)"\s*:\s*"([^"]+)"[^{}]*\}', text)
    if dict_m:
        try:
            return json.loads(dict_m.group(0))
        except Exception:
            pass

    return None

def resolve_model_id(model_id: str):
    try:
        AutoTokenizer.from_pretrained(model_id)
        return model_id
    except Exception as e:
        err_str = str(e).lower()
        if ("gated" in err_str or "401" in err_str or "restricted" in err_str) and "google/gemma-2-2b-it" in model_id:
            print("Notice: 'google/gemma-2-2b-it' requires gated authorization and HF_TOKEN was not detected.")
            print("Seamlessly falling back to identical public architecture mirror: 'unsloth/gemma-2-2b-it'...")
            return "unsloth/gemma-2-2b-it"
        raise e

def evaluate_model_pipeline(model_id: str, samples: list):
    resolved_id = resolve_model_id(model_id)
    print(f"\n=======================================================")
    print(f" STARTING EVALUATION: {resolved_id} (Target: {model_id})")
    print(f"=======================================================")
    
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
    print_telemetry(f"Loading {resolved_id}")
    
    tokenizer = AutoTokenizer.from_pretrained(resolved_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    model = AutoModelForCausalLM.from_pretrained(
        resolved_id,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
    model.eval()
    
    print_telemetry(f"{resolved_id} Resident in Memory")
    
    metrics = {
        "BFCL": {"invoked": 0, "correct_tool": 0, "valid_args": 0, "total": 0, "latencies": []},
        "ToolBench": {"invoked": 0, "correct_tool": 0, "valid_args": 0, "total": 0, "latencies": []}
    }
    sample_traces = []
    
    try:
        pbar = tqdm(samples, desc=f"Eval: {model_id.split('/')[-1]}", unit="sample")
        for item in pbar:
            b_name = item["benchmark"]
            metrics[b_name]["total"] += 1
            expected = item["expected_tool"]
            
            t0 = time.time()
            chat = [{"role": "user", "content": item["prompt"]}]
            formatted = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(formatted, return_tensors="pt", truncation=True, max_length=1536).to(model.device)
            
            with torch.no_grad():
                out_ids = model.generate(
                    **inputs,
                    max_new_tokens=100,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id
                )
            latency = time.time() - t0
            metrics[b_name]["latencies"].append(latency)
            
            gen_text = tokenizer.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
            parsed = parse_agent_call(gen_text)
            
            invoked = parsed is not None
            matched = False
            has_args = False
            
            if invoked:
                metrics[b_name]["invoked"] += 1
                name = str(parsed.get("name", "")).strip().lower()
                if expected in name or name in expected:
                    matched = True
                    metrics[b_name]["correct_tool"] += 1
                if parsed.get("arguments") or parsed.get("parameters"):
                    has_args = True
                    metrics[b_name]["valid_args"] += 1
                    
            sample_traces.append({
                "id": item["id"],
                "benchmark": b_name,
                "expected": expected,
                "generated": gen_text,
                "parsed": parsed,
                "invoked": invoked,
                "matched": matched,
                "latency_sec": latency
            })
            
            # Dynamic live telemetry in tqdm postfix
            tel = get_system_telemetry()
            vram_str = f"{tel['gpus'][0]['alloc_gb']:.1f}GB" if tel['gpus'] else "CPU"
            pbar.set_postfix({
                "VRAM": vram_str,
                "Invoked": f"{metrics[b_name]['invoked']}/{metrics[b_name]['total']}",
                "Matched": f"{metrics[b_name]['correct_tool']}"
            })
            
    finally:
        print(f"\nPurging {model_id} from VRAM...")
        del model
        del tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print_telemetry(f"{model_id} Purged")
        
    return metrics, sample_traces

# Execute Base Model Evaluation
BASE_ID = "google/gemma-2-2b-it"
SFT_ID = "orangefabercastell/gemma-2-2b-it-pi-mono-sft"

base_metrics, base_traces = evaluate_model_pipeline(BASE_ID, MASTER_EVAL_SUITE)
sft_metrics, sft_traces = evaluate_model_pipeline(SFT_ID, MASTER_EVAL_SUITE)


In [ ]:
# [Cell 4] Empirical Scorecard, Visual Analytics & Artifact Export
print_telemetry("Synthesizing Final Benchmark Results")

rows = []
for bench in ["BFCL", "ToolBench"]:
    b_tot = base_metrics[bench]["total"]
    s_tot = sft_metrics[bench]["total"]
    
    b_inv = (base_metrics[bench]["invoked"] / b_tot) * 100
    s_inv = (sft_metrics[bench]["invoked"] / s_tot) * 100
    
    b_match = (base_metrics[bench]["correct_tool"] / b_tot) * 100
    s_match = (sft_metrics[bench]["correct_tool"] / s_tot) * 100
    
    b_args = (base_metrics[bench]["valid_args"] / b_tot) * 100
    s_args = (sft_metrics[bench]["valid_args"] / s_tot) * 100
    
    b_lat = float(np.mean(base_metrics[bench]["latencies"]))
    s_lat = float(np.mean(sft_metrics[bench]["latencies"]))
    
    rows.extend([
        {"Benchmark": bench, "Metric": "Tool Invocation Rate (%)", "Base Model": b_inv, "SFT Agent Model": s_inv, "Delta": s_inv - b_inv},
        {"Benchmark": bench, "Metric": "Tool Selection Accuracy (%)", "Base Model": b_match, "SFT Agent Model": s_match, "Delta": s_match - b_match},
        {"Benchmark": bench, "Metric": "Valid Arguments Rate (%)", "Base Model": b_args, "SFT Agent Model": s_args, "Delta": s_args - b_args},
        {"Benchmark": bench, "Metric": "Mean Latency (s/sample)", "Base Model": b_lat, "SFT Agent Model": s_lat, "Delta": s_lat - b_lat}
    ])

scorecard_df = pd.DataFrame(rows)

# 1. Print Standard Formatted Scorecard
print("=" * 86)
print("             OUT-OF-DOMAIN AGENTIC BENCHMARK SCORECARD (200 SAMPLES)")
print("=" * 86)
print(f"{'Benchmark':<12} | {'Metric':<28} | {'Base':<8} | {'SFT':<8} | {'Delta':<8}")
print("-" * 86)
for _, r in scorecard_df.iterrows():
    if "Latency" in r["Metric"]:
        print(f"{r['Benchmark']:<12} | {r['Metric']:<28} | {r['Base Model']:<8.2f} | {r['SFT Agent Model']:<8.2f} | {r['Delta']:<+8.2f}s")
    else:
        print(f"{r['Benchmark']:<12} | {r['Metric']:<28} | {r['Base Model']:<7.1f}% | {r['SFT Agent Model']:<7.1f}% | {r['Delta']:<+7.1f}%")
print("=" * 86)

# 2. Persist Artifacts
with open("benchmark_results_raw.json", "w") as f:
    json.dump({"base_traces": base_traces, "sft_traces": sft_traces}, f, indent=2)

scorecard_df.to_csv("benchmark_scorecard_summary.csv", index=False)
print("\nArtifacts Saved Successfully:")
print("  - benchmark_results_raw.json (400 execution trajectories)")
print("  - benchmark_scorecard_summary.csv (Aggregated empirical metrics)")

print_telemetry("All Benchmark Tasks Finished")

# 3. Styled Display
scorecard_df.style.format({
    "Base Model": "{:.2f}",
    "SFT Agent Model": "{:.2f}",
    "Delta": "{:+.2f}"
}).set_properties(**{'text-align': 'center'})
